# 🔁 Repaso práctico · Bloques 1–5 de EDA
### Bloque 6 · Tema 1 · Sesión 1 de 2 · Parte A: repaso práctico (≈ 1 h 30 min)

> **Objetivo:** antes de entrar en el caso *Housing Madrid*, **practicar con código** las herramientas de los Bloques 1 a 5 de la guía. Todo en un único notebook, con un dataset pequeño y conocido (50 tiendas + 1.200 pedidos de una cadena de CLM).

| ⏱️ Min | Parte | Bloque de la guía | Qué haremos |
|---|---|---|---|
| 14 | 🎬 Calentamiento | Bloque 0 | Vídeo (5) + radiografía del dataset (9) |
| 8 | 🧮 Del negocio al estadístico | Bloque 1 | Ticket medio, KPI, tasa de variación |
| 14 | 📊 Univariante | Bloque 2 | Media vs mediana, percentiles, CV, asimetría |
| 18 | 🔗 Correlación | Bloque 3 | Pearson vs Spearman, Anscombe, variable oculta |
| 5 | ☕ Pausa + recurso | — | Lectura amena |
| 12 | 🚨 Outliers | Bloque 4 | IQR, z-score, MAD y **decisión** |
| 10 | 📈 Visualización | Bloque 5 | Histograma, KDE, violín, gráficos engañosos |
| 4 | 🧭 Cierre | Bloque 6 | La receta de 7 pasos → puente al caso Housing |
| 5 | 🎯 Quiz relámpago | 1–5 | 5 preguntas |

**Leyenda:** 🎬 vídeo (inglés) · 📖 lectura (inglés) · 🧠 mini-reto · ✍️ tu respuesta · `# TODO` = completa tú el código.

> ℹ️ **Equivalencia de numeración:** el notebook de clase `EDA_BI_Integrado` numera los bloques del 1 al 7; aquí seguimos la numeración de la **Guía Docente** (Bloque 0 a Bloque 6).

In [ ]:
# ============================================================
# CELDA 0 · Entorno y dataset de clase (ejecútala primero)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Semilla fija: TODOS veremos los mismos números
np.random.seed(7)

# --- Dataset 1: 50 tiendas ------------------------------------
tiendas = pd.DataFrame({
    "tienda":       [f"T{i:02d}" for i in range(1, 51)],
    "ciudad":       np.random.choice(["Albacete","Ciudad Real","Cuenca","Guadalajara","Toledo"], 50),
    "m2":           np.random.randint(80, 400, 50),
    "empleados":    np.random.randint(2, 15, 50),
    "ventas_mes":   np.round(np.random.lognormal(mean=9.2, sigma=0.5, size=50), 0),
    "devoluciones": np.random.randint(0, 30, 50),
})
# Un outlier "real" a propósito: la tienda T18 con ventas disparadas
tiendas.loc[17, "ventas_mes"] = 1_250_000   # ¿error, evento real o señal de negocio?

# --- Dataset 2: 1.200 pedidos ---------------------------------
pedidos = pd.DataFrame({
    "pedido":     np.arange(1, 1201),
    "importe":    np.round(np.random.lognormal(mean=3.6, sigma=0.7, size=1200), 2),
    "categoria":  np.random.choice(["Electrónica","Moda","Hogar","Alimentación"], 1200,
                                   p=[0.25, 0.35, 0.20, 0.20]),
    "envio_h":    np.round(np.random.gamma(shape=2.5, scale=8, size=1200), 1),
    "devuelto":   np.random.choice([0, 1], 1200, p=[0.92, 0.08]),
})
# 10 pedidos duplicados a propósito (para practicar limpieza)
pedidos = pd.concat([pedidos, pedidos.sample(10, random_state=1)], ignore_index=True)

print("✅ Tiendas:", tiendas.shape, "| Pedidos:", pedidos.shape)
tiendas.head()

---
## 🎬 Calentamiento (14 min)

**🎬 Vídeo (4 min 48 s, inglés):** *Hans Rosling's 200 Countries, 200 Years, 4 Minutes* (BBC, *The Joy of Stats*) → https://www.youtube.com/watch?v=jbkSRLYSojo

Mientras lo ves, anota en 2 líneas:
1. ¿Qué **variables** se cruzan en el gráfico y qué escala usa cada eje?
2. ¿Qué **valores atípicos / saltos** ves (guerras, epidemias)? ¿Los llamarías *error*, *evento* o *señal*?

> Esa pregunta —*¿error, evento o señal?*— es la que repetiremos todo el día.

### Bloque 0 · Radiografía del dataset (head · info · describe)
🧠 **Mini-reto (9 min):** inspecciona los datos y **limpia los duplicados**. ¿Cambian los KPI al limpiar?

In [ ]:
tiendas.info()
print("\nNulos por columna:\n", tiendas.isna().sum())
display(tiendas.describe().T)

# Calidad: duplicados en pedidos
n_dup = pedidos.duplicated().sum()
print(f"\nPedidos duplicados: {n_dup}")
# TODO 1: crea 'pedidos_limpio' sin duplicados y con el índice reiniciado
#   pista: .drop_duplicates()  y  .reset_index(drop=True)
pedidos_limpio = ____
print("Filas antes/después de limpiar:", len(pedidos), "→", len(pedidos_limpio))

# ¿Cómo se reparten las tiendas y las ventas por ciudad?
# TODO 2: agrupa por 'ciudad' y calcula count, mean y median de 'ventas_mes'
#   pista: .groupby(...)[...].agg([...])
____

---
## 🧮 Bloque 1 · Del negocio al estadístico (8 min)
Una pregunta de negocio ("¿cuánto gasta un cliente?") se traduce en **una medida concreta**. Y esa medida cambia según la limpieza y el segmento.

🧠 **Mini-reto:** calcula el **ticket medio** con y sin duplicados, y la **tasa de variación** media entre dos meses para tiendas grandes (> 200 m²) frente a pequeñas.

In [ ]:
ticket_con_dup = pedidos["importe"].mean()
# TODO 1: ticket medio SIN duplicados (usa 'pedidos_limpio')
ticket_limpio  = ____
print(f"Ticket medio CON duplicados : {ticket_con_dup:.2f} €")
print(f"Ticket medio SIN duplicados : {ticket_limpio:.2f} €  (diferencia {ticket_limpio - ticket_con_dup:+.2f} €)")
# TODO 2: ticket medio Y mediano por categoría
____

# KPI: tasa de variación entre dos meses (simulamos el mes 2)
np.random.seed(11)
ventas_mes2 = tiendas["ventas_mes"] * np.random.uniform(0.85, 1.20, len(tiendas))
# TODO 3: tasa de variación (%) = (mes2 - mes1) / mes1 * 100
tiendas["tasa_var"] = ____
tiendas["tamano"] = np.where(tiendas["m2"] > 200, "grande", "pequeña")
# TODO 4: tasa de variación media por 'tamano' (grande vs pequeña)
____

---
## 📊 Bloque 2 · Estadística descriptiva univariante (14 min)
Posición (media, mediana, percentiles), dispersión (desviación típica, **CV**) y forma (**asimetría**).

🧠 **Mini-reto:** ¿qué cifra usarías para **negociar** un convenio y cuál para **presupuestar**? ¿Cuál de las dos tiendas es relativamente más volátil?

In [ ]:
salarios = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 25000]
# TODO 1: calcula media y mediana de 'salarios' con numpy
media, mediana = ____, ____
print(f"Media  : {media:>9,.0f} €  ← ¿qué la estira?")
print(f"Mediana: {mediana:>9,.0f} €  ← ¿a quién representa?")

# Percentil 90 del tiempo de envío → SLA "90 % de pedidos en ≤ 48 h"
# TODO 2: P90 de 'envio_h' en pedidos_limpio  (pista: np.percentile(serie, 90))
p90 = ____
print(f"\nP90 envío: {p90:.1f} h → {'✅ cumple SLA' if p90 <= 48 else '❌ incumple SLA'}")

# Coeficiente de variación: comparar dispersión entre magnitudes distintas
np.random.seed(3)
A = np.random.normal(1000, 50, 100)      # tienda A
B = np.random.normal(10000, 400, 100)    # tienda B
# TODO 3: define cv(x) = desviación típica muestral (ddof=1) / media * 100
cv = lambda x: ____
print(f"CV tienda A: {cv(A):.1f} %  |  CV tienda B: {cv(B):.1f} %")
print(f"Desv. típica A: {np.std(A, ddof=1):.0f} €  |  B: {np.std(B, ddof=1):.0f} €  ← ¿por qué engaña?")

# TODO 4: asimetría (skewness) del importe de pedidos_limpio
print(f"\nAsimetría del importe: {____:.2f} (>0 = cola a la derecha)")

---
## 🔗 Bloque 3 · Relaciones entre variables (18 min)
**Mira antes de calcular.** Un coeficiente resume *una sola* cosa y puede engañar.

**🎬 Para abrir el apetito (5 min, inglés):** *Same Stats, Different Graphs* (Autodesk Research) — 12 conjuntos de datos con **la misma media, desviación y correlación** pero formas radicalmente distintas (¡incluido un dinosaurio!) → https://www.research.autodesk.com/publications/same-stats-different-graphs/

🧠 **Mini-reto 1:** ¿cuánto vale la correlación m² ↔ ventas **con y sin** la tienda T18? ¿Y con Spearman?

In [ ]:
cols = ["m2", "empleados", "ventas_mes", "devoluciones"]

# TODO 1: scatterplot de m2 (x) vs ventas_mes (y), coloreado por ciudad
____
plt.title("Superficie vs ventas mensuales (¿ves a T18?)")
plt.show()

# TODO 2: crea 'sin_t18' (todas las tiendas menos T18)
sin_t18 = ____
print("r Pearson  m2~ventas  CON T18:", round(tiendas["m2"].corr(tiendas["ventas_mes"]), 3))
# TODO 3: la misma correlación SIN T18
print("r Pearson  m2~ventas  SIN T18:", ____)
# TODO 4: correlación de SPEARMAN CON T18  (pista: .corr(otra, method="spearman"))
print("r Spearman m2~ventas  CON T18:", ____)

sns.heatmap(sin_t18[cols].corr(method="spearman"), annot=True, cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlación de Spearman (sin T18)")
plt.show()

🧠 **Mini-reto 2 · El cuarteto de Anscombe (4 datasets, misma estadística).** Calcula la media de *y*, la correlación y la recta de ajuste de cada uno… y **luego dibújalos**.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
ans = pd.DataFrame({
    "x": x123*3 + [8,8,8,8,8,8,8,19,8,8,8],
    "y": [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68,
          9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74,
          7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73,
          6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89],
    "dataset": np.repeat(["I","II","III","IV"], 11),
})
# TODO 1: por 'dataset' calcula la media de y y la correlación x~y
#   pista: ans.groupby("dataset").apply(lambda g: ...)  o un bucle for
____

# TODO 2: dibuja los 4 datasets (pista: sns.lmplot con col="dataset")
____
plt.show()
# ✍️ Describe qué ves en cada panel: recta / curva / outlier / palanca

🧠 **Mini-reto 3 · La variable oculta.** Dos magnitudes pueden moverse juntas porque **una tercera** las empuja. ¿Qué ocurre con la correlación si comparamos solo días con temperaturas parecidas?

📖 **Lectura optativa (inglés, 5 min):** Tyler Vigen, *Spurious Correlations* → https://www.tylervigen.com/spurious-correlations

In [ ]:
np.random.seed(5)
temp = np.random.uniform(10, 38, 200)                         # temperatura (la variable oculta)
helados  = 20 * temp + np.random.normal(0, 60, 200)           # ventas de helados
rescates = 0.8 * temp + np.random.normal(0, 5, 200)           # rescates en playa
df_sp = pd.DataFrame({"temp": temp, "helados": helados, "rescates": rescates})

# TODO 1: correlación global entre helados y rescates
print("r(helados, rescates) global:", ____)

# Controlamos por la tercera variable: correlación DENTRO de cada tramo de temperatura
df_sp["tramo"] = pd.qcut(df_sp["temp"], 4, labels=["frío", "templado", "cálido", "caluroso"])
# TODO 2: correlación helados~rescates dentro de cada 'tramo'
#   pista: groupby("tramo", observed=True)[[...]].apply(lambda g: g.corr().iloc[0, 1])
____
# ✍️ ¿Comen helados los socorristas... o hace calor? ¿Qué pasa con r dentro de cada tramo?

---
## ☕ Pausa activa (5 min) · 📖 Lectura amena
Mientras te estiras: **¿puede un supermercado saber que una clienta está embarazada solo con sus compras?**
- 📖 Forbes: *How Target Figured Out A Teen Girl Was Pregnant Before Her Father Did* → https://www.forbes.com/sites/kashmirhill/2012/02/16/how-target-figured-out-a-teen-girl-was-pregnant-before-her-father-did/
- 📖 …y el contrapunto crítico: KDnuggets, *Did Target Really Predict a Teen's Pregnancy? The Inside Story* → https://www.kdnuggets.com/2014/05/target-predict-teen-pregnancy-inside-story.html

💬 Cuando volvamos: ¿qué parte de la historia es **correlación**, qué parte **causalidad** y qué parte **titular**?

---
## 🚨 Bloque 4 · Caza de outliers (12 min)
Cuatro métodos, una pregunta: **¿error, evento o señal?** Detectar es fácil; **decidir** es el trabajo del analista.

🧠 **Mini-reto:** aplica IQR, z-score y MAD a `ventas_mes`. ¿Coinciden? Compara la media con y sin T18 y **justifica** tu decisión.

In [ ]:
v = tiendas["ventas_mes"]

# 1) Regla del IQR: fuera de [Q1 - 1,5·IQR, Q3 + 1,5·IQR]
Q1, Q3 = np.percentile(v, [25, 75])
# TODO 1: calcula IQR y el límite superior
IQR = ____
lim_sup = ____
print(f"IQR: límite superior = {lim_sup:,.0f} € → outliers:", tiendas.loc[v > lim_sup, "tienda"].tolist())

# 2) z-score: |z| > 3
# TODO 2: z = (valor - media) / desviación típica
z = ____
print("z-score (|z|>3)            → outliers:", tiendas.loc[z.abs() > 3, "tienda"].tolist())

# 3) MAD (robusto): |v - mediana| / (1,4826 · MAD) > 3
med = np.median(v)
# TODO 3: MAD = mediana de |v - med|
mad = ____
mz = np.abs(v - med) / (1.4826 * mad)
print("MAD (|z robusto|>3)        → outliers:", tiendas.loc[mz > 3, "tienda"].tolist())

# 4) Impacto en las medidas
# TODO 4: media sin T18
media_sin = ____
print(f"\nMedia CON T18: {v.mean():,.0f} €  |  SIN T18: {media_sin:,.0f} €  |  Mediana: {v.median():,.0f} €")

sns.boxplot(y=v); plt.title("¿Ves el outlier de la tienda T18?"); plt.show()
# ✍️ Decide y JUSTIFICA: ¿eliminar, corregir, conservar o analizar aparte?

---
## 📈 Bloque 5 · Visualización de distribuciones (10 min)
**🎬 Vídeo (4 min, inglés):** TED-Ed, *How to spot a misleading graph* (Lea Gaslowitz) → https://www.ted.com/talks/lea_gaslowitz_how_to_spot_a_misleading_graph

🧠 **Mini-reto:** (1) cómo cambia el histograma con el nº de *bins*; (2) violín por categoría; (3) **el mismo dato, dos gráficos**: uno honesto y uno engañoso.

In [ ]:
imp = pedidos_limpio["importe"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
# TODO 1: dibuja un histograma de 'imp' con 5, 20 y 60 bins (uno por panel)
for ax, b in zip(axes, [5, 20, 60]):
    ____
    ax.set_title(f"{b} bins")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# TODO 2: violín del importe por categoría (axes[0])  y  histograma con KDE de log(importe) (axes[1])
____
____
plt.tight_layout(); plt.show()

# El mismo dato, dos mensajes: ticket medio por categoría
medias = pedidos_limpio.groupby("categoria")["importe"].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
medias.plot.bar(ax=axes[0], color="#E67E22"); axes[0].set_ylim(medias.min() - 2, medias.max() + 1)
axes[0].set_title("Eje truncado")
# TODO 3: la versión HONESTA: eje y desde 0
medias.plot.bar(ax=axes[1], color="#2E86C1"); ____
axes[1].set_title("Eje desde 0")
plt.tight_layout(); plt.show()
# ✍️ ¿Qué forma tiene el importe? ¿Qué historia de negocio cuenta? ¿Cuál es el truco del gráfico naranja?

---
## 🧭 Cierre · La receta de EDA de 7 pasos (Bloque 6)
Todo lo que hemos hecho encaja en **una receta reproducible**. En *Housing* y *Santander* la aplicaremos de principio a fin.

| Paso | Qué | Hoy lo hemos visto en… |
|---|---|---|
| 1 | Cargar | Setup |
| 2 | Inspeccionar | Bloque 0 |
| 3 | Limpiar | Bloque 0 (duplicados) |
| 4 | Univariante | Bloque 2 |
| 5 | Bivariante | Bloque 3 |
| 6 | Outliers | Bloque 4 |
| 7 | Conclusiones | …lo que haremos en los casos |

🧠 **Último reto:** completa la función `receta_eda` con los pasos 5 y 6 mínimos.

In [ ]:
def receta_eda(df, objetivo=None):
    print("1) FORMA      :", df.shape)
    print("2) NULOS      :", int(df.isna().sum().sum()), "| DUPLICADOS:", int(df.duplicated().sum()))
    num = df.select_dtypes("number")
    print("4) UNIVARIANTE (asimetría):", num.skew().round(2).to_dict())
    if objetivo is not None and objetivo in num:
        # TODO 5) bivariante: correlación de Spearman de cada columna con 'objetivo'
        print(f"5) BIVARIANTE  (Spearman con '{objetivo}'):")
        print(____)
    # TODO 6) outliers: cuenta por columna los valores fuera de [Q1-1,5·IQR, Q3+1,5·IQR]
    Q1, Q3 = num.quantile(0.25), num.quantile(0.75); IQR = Q3 - Q1
    n_out = ____
    print("6) OUTLIERS (IQR) por columna:", n_out.to_dict())
    return df

_ = receta_eda(tiendas[["m2", "empleados", "ventas_mes", "devoluciones"]], objetivo="ventas_mes")

---
## 🎯 Quiz relámpago (5 min) · Bloques 1–5
Responde en 30 segundos cada una (en parejas, sin ordenador):

1. Un sueldo de 25.000 € en una plantilla de 1.500 € → ¿qué medida usas para *presupuestar* y cuál para *negociar*?
2. La tienda A tiene σ = 50 € y media 1.000 €; la B, σ = 400 € y media 10.000 €. ¿Cuál es más volátil **en términos relativos**?
3. r = 0,85 entre ventas de helados y rescates en playa. ¿Puedes concluir que los helados causan rescates? ¿Qué harías?
4. Un único punto extremo, ¿puede cambiar el Pearson? ¿Y el Spearman?
5. Un gráfico de barras con el eje y empezando en 98 en lugar de 0: ¿cuándo es engañoso?